# DataSage 3/4 — Answering Inference

Loads the **answering** GRPO LoRA adapter and runs inference across all domains and personas.  
Also runs the **base model** (no LoRA) on the same seeds for comparison.

| Setting | Value |
|---------|-------|
| Runtime | **GPU** (T4 minimum, A100 recommended) |
| Secrets | `HF_TOKEN` (optional, models are public) |
| LoRA adapter | `ricalanis/answering-grpo` |
| Base model | `unsloth/Qwen2.5-3B-Instruct` |
| Environment | `https://ricalanis-datasage-answering.hf.space` |
| Output | `answering_results.json` |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/03_answering_inference.ipynb)

In [ ]:
# Cell 1: Install dependencies
!pip install -q unsloth trl requests numpy datasets huggingface_hub

In [ ]:
# Cell 2: Imports, Colab detection, GPU check
import os
import sys
import json
import re
import time
import requests
import numpy as np
import torch
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("Loaded HF_TOKEN from Colab Secrets")
    except Exception:
        print("HF_TOKEN not in Secrets (OK — models are public)")
else:
    print("Running locally")

assert torch.cuda.is_available(), (
    "No GPU detected! Go to: Runtime > Change runtime type > GPU (T4 or better)"
)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Cell 3: All configuration

BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
LORA_REPO  = "ricalanis/answering-grpo"
MAX_SEQ_LENGTH = 2048

ENV_URL = "https://ricalanis-datasage-answering.hf.space"

DOMAINS  = ["hr", "sales", "pm", "it_ops"]
PERSONAS = ["executive", "manager", "ic"]
# Answering is single-step, so we run 1 episode per (domain, persona) pair
# = 4 domains x 3 personas = 12 episodes

SYSTEM_PROMPT = (
    "You are a data analyst answering questions for a specific persona.\n"
    "Adapt your language and depth:\n"
    "- Executive: strategic, financial impact, KPIs\n"
    "- Manager: operational, actionable recommendations\n"
    "- IC: plain language, personal impact\n\n"
    "Cite specific data columns. Be data-grounded. Respond with JSON:\n"
    '{"answer": "<your answer>", "cited_columns": ["col1", "col2"], "reasoning": "<brief>"}'
)

print("Config loaded.")
print(f"  LoRA:     {LORA_REPO}")
print(f"  Env:      {ENV_URL}")
print(f"  Domains:  {DOMAINS}")
print(f"  Personas: {PERSONAS}")
print(f"  Total episodes: {len(DOMAINS) * len(PERSONAS)} per model")

In [ ]:
# Cell 4: Helper functions

import json
import re
import time
import requests
import numpy as np
import torch


def parse_action(text):
    """Parse model output into an answering action dict."""
    m = re.search(r'\{[^{}]*"answer"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "answer" in data:
                return data
        except json.JSONDecodeError:
            pass
    cited = re.findall(r'\b([A-Z][a-zA-Z]+(?:[A-Z][a-z]+)*)\b', text)
    return {"answer": text, "cited_columns": cited[:5], "reasoning": ""}


def env_reset(seed=42):
    try:
        r = requests.post(f"{ENV_URL}/web/reset", json={"seed": seed}, timeout=30)
        r.raise_for_status()
        d = r.json()
        return d.get("observation", d)
    except Exception as e:
        print(f"  [WARN] env_reset failed: {e}")
        return {"error": str(e)}


def env_step(action):
    try:
        r = requests.post(f"{ENV_URL}/web/step", json={"action": action}, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  [WARN] env_step failed: {e}")
        return {"error": str(e), "reward": 0, "done": True, "observation": {}, "info": {}}


def format_obs(obs):
    return (
        f"Domain: {obs.get('domain', 'unknown')}\n"
        f"Persona: {obs.get('persona', 'unknown')} - {obs.get('persona_description', '')}\n"
        f"Question: {obs.get('question', 'N/A')}\n"
        f"Available Columns: {obs.get('available_columns', [])}\n"
        f"Column Stats: {str(obs.get('column_stats', ''))[:800]}\n"
        f"Dataset Summary: {str(obs.get('dataset_summary', ''))[:400]}\n\n"
        "Answer the question in the persona's style. Output JSON."
    )


def generate(model, tokenizer, sys_prompt, user_prompt, max_new_tokens=256):
    messages = [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids=input_ids, max_new_tokens=max_new_tokens,
            temperature=0.7, do_sample=True, pad_token_id=tokenizer.pad_token_id,
        )
    new_tokens = output_ids[0][input_ids.shape[1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)
    response = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL).strip()
    return response


def run_episode(model, tokenizer, seed=42):
    """Single-step answering episode."""
    obs = env_reset(seed)
    if "error" in obs:
        return {"error": obs["error"]}
    resp = generate(model, tokenizer, SYSTEM_PROMPT, format_obs(obs))
    action = parse_action(resp)
    result = env_step(action)
    info = result.get("info", {})
    return {
        "reward": float(result.get("reward", 0)),
        "faithfulness": float(info.get("faithfulness", 0)),
        "persona_alignment": float(info.get("persona_alignment", 0)),
        "domain": obs.get("domain", "unknown"),
        "persona": obs.get("persona", "unknown"),
        "question": obs.get("question", ""),
        "answer": action.get("answer", resp),
        "cited_columns": action.get("cited_columns", []),
    }


try:
    r = requests.get(ENV_URL, timeout=10)
    print(f"Environment OK: {ENV_URL} (status {r.status_code})")
except Exception as e:
    print(f"WARNING: Environment UNREACHABLE: {ENV_URL} ({e})")

print("All helpers defined.")

In [ ]:
# Cell 5: Load LoRA model
from unsloth import FastLanguageModel

print(f"Loading LoRA adapter: {LORA_REPO}")
t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=LORA_REPO, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)
print(f"Loaded in {time.time()-t0:.1f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 6: Run LoRA inference — 4 domains x 3 personas = 12 episodes
print("=" * 60)
print(f"ANSWERING — DataSage LoRA ({LORA_REPO})")
print(f"  {len(DOMAINS)} domains x {len(PERSONAS)} personas = {len(DOMAINS)*len(PERSONAS)} episodes")
print("=" * 60)

lora_results = []
seed_counter = 1
for domain in DOMAINS:
    for persona in PERSONAS:
        seed = seed_counter
        seed_counter += 1
        print(f"  [{domain}] {persona} seed={seed}", end=" ... ")
        t0 = time.time()
        result = run_episode(model, tokenizer, seed)
        result["seed"] = seed
        result["latency_s"] = round(time.time() - t0, 1)
        lora_results.append(result)
        if "error" in result:
            print(f"ERROR: {result['error']}")
        else:
            print(f"R={result['reward']:.4f} faith={result['faithfulness']:.4f} PA={result['persona_alignment']:.4f} ({result['latency_s']}s)")

valid = [r for r in lora_results if "error" not in r]
if valid:
    print(f"\n--- LoRA Summary ({len(valid)} episodes) ---")
    print(f"  Mean Reward: {np.mean([r['reward'] for r in valid]):.4f}")
    for p in PERSONAS:
        pr = [r for r in valid if r.get("persona") == p]
        if pr:
            print(f"  {p}: R={np.mean([r['reward'] for r in pr]):.4f} ({len(pr)} eps)")

del model, tokenizer
torch.cuda.empty_cache()
print(f"\nGPU freed. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB")

In [ ]:
# Cell 7: Load base model
from unsloth import FastLanguageModel

print(f"Loading base model: {BASE_MODEL}")
t0 = time.time()
base_m, base_t = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL, max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True, fast_inference=True, max_lora_rank=16, gpu_memory_utilization=0.6,
)
if base_t.pad_token is None:
    base_t.pad_token = base_t.eos_token
FastLanguageModel.for_inference(base_m)
print(f"Loaded in {time.time()-t0:.1f}s")

In [ ]:
# Cell 8: Run base model — same domain x persona combos
print(f"\nBASE MODEL — Answering ({BASE_MODEL})")

base_results = []
seed_counter = 1
for domain in DOMAINS:
    for persona in PERSONAS:
        seed = seed_counter
        seed_counter += 1
        print(f"  [{domain}] {persona} seed={seed}", end=" ... ")
        result = run_episode(base_m, base_t, seed)
        result["seed"] = seed
        base_results.append(result)
        if "error" in result:
            print(f"ERROR: {result['error']}")
        else:
            print(f"R={result['reward']:.4f}")

del base_m, base_t
torch.cuda.empty_cache()
print("Done. GPU freed.")

In [ ]:
# Cell 9: Summarize and save
import os
import sys
import json
import numpy as np
from datetime import datetime

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")


def summarize(results):
    valid = [r for r in results if "error" not in r]
    if not valid:
        return {"error": "no valid episodes"}
    return {
        "reward_mean": float(np.mean([r["reward"] for r in valid])),
        "reward_std": float(np.std([r["reward"] for r in valid])),
        "faithfulness_mean": float(np.mean([r["faithfulness"] for r in valid])),
        "persona_alignment_mean": float(np.mean([r["persona_alignment"] for r in valid])),
        "n_episodes": len(valid),
        "per_persona": {
            p: {
                "reward_mean": float(np.mean([r["reward"] for r in valid if r.get("persona") == p])),
                "n": len([r for r in valid if r.get("persona") == p]),
            }
            for p in PERSONAS
            if any(r.get("persona") == p for r in valid)
        },
        "per_domain": {
            d: {
                "reward_mean": float(np.mean([r["reward"] for r in valid if r.get("domain") == d])),
                "n": len([r for r in valid if r.get("domain") == d]),
            }
            for d in DOMAINS
            if any(r.get("domain") == d for r in valid)
        },
        "per_episode": [
            {
                "domain": r.get("domain"),
                "persona": r.get("persona"),
                "reward": float(r["reward"]),
                "faithfulness": float(r.get("faithfulness", 0)),
                "persona_alignment": float(r.get("persona_alignment", 0)),
            }
            for r in valid
        ],
    }


output = {
    "task": "answering",
    "datasage_lora": summarize(lora_results),
    "base_model": summarize(base_results),
    "raw_lora": [
        {k: v for k, v in r.items() if k not in ("answer",)}
        for r in lora_results
    ],
    "raw_base": [
        {k: v for k, v in r.items() if k not in ("answer",)}
        for r in base_results
    ],
    "metadata": {
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "lora_repo": LORA_REPO, "base_model": BASE_MODEL,
        "env_url": ENV_URL,
        "domains": DOMAINS, "personas": PERSONAS,
    },
}

OUTPUT_FILE = "answering_results.json"
with open(OUTPUT_FILE, "w") as f:
    json.dump(output, f, indent=2)

print(f"Saved: {OUTPUT_FILE} ({os.path.getsize(OUTPUT_FILE)/1024:.1f} KB)")

ls = output["datasage_lora"]
bs = output["base_model"]
if "error" not in ls and "error" not in bs:
    print(f"\n  LoRA  R={ls['reward_mean']:.4f}  faith={ls['faithfulness_mean']:.4f}  ({ls['n_episodes']} eps)")
    print(f"  Base  R={bs['reward_mean']:.4f}  faith={bs['faithfulness_mean']:.4f}  ({bs['n_episodes']} eps)")
    print(f"  Delta R={ls['reward_mean']-bs['reward_mean']:+.4f}")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print(f"\nDownloaded {OUTPUT_FILE} to your browser.")